In [11]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!unzip -q "/content/drive/MyDrive/KhPI_HW/linear_algebra_intro_practice-main.zip" -d /content/idpp
!find /content/idpp -name "vectors.py"
!find /content/idpp -name "pyproject.toml"

replace /content/idpp/linear_algebra_intro_practice-main/.gitignore? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/idpp/linear_algebra_intro_practice-main/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/idpp/linear_algebra_intro_practice-main/matrices.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/idpp/linear_algebra_intro_practice-main/matrix_decomposition.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/idpp/linear_algebra_intro_practice-main/matrix_mapping.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/idpp/linear_algebra_intro_practice-main/pyproject.toml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/idpp/linear_algebra_intro_practice-main/regularization.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/idpp/linear_algebra_intro_practice-main/tutorial_gmm.ipynb? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/idpp/linear_algebra_intro_practice-main/tutorial_pca.ipynb? [y]es, [n]o,

In [13]:
%cd /content/idpp/linear_algebra_intro_practice-main

/content/idpp/linear_algebra_intro_practice-main


In [15]:
!pip -q install uv
!uv sync
!uv run pytest tests/test_vectors.py -v

Resolved 113 packages in 1ms
Checked 109 packages in 1ms
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- /content/idpp/linear_algebra_intro_practice-main/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/idpp/linear_algebra_intro_practice-main
configfile: pyproject.toml
plugins: platformdirs-4.12.1, anyio-4.15.1
collected 0 items                                                              

============================ no tests ran in 0.01s =============================
ERROR: file or directory not found: tests/test_vectors.py



In [20]:
# Перезавантажуємо модуль, щоб Colab точно взяв останню версію vectors.py
import importlib
import numpy as np
from scipy import sparse
import vectors

importlib.reload(vectors)


def test_get_vector_shape_and_type():
    # Перевіряємо, що створюється саме NumPy-вектор-стовпець розміру (dim, 1)
    result = vectors.get_vector(7)

    assert isinstance(result, np.ndarray), "get_vector має повертати np.ndarray"
    assert result.shape == (7, 1), (
        f"Очікувалася форма (7, 1), отримано {result.shape}"
    )


def test_get_sparse_vector_shape_and_format():
    # Перевіряємо форму та формат COO (в умові sparse.coo_matrix), який прямо вказаний у типі повернення
    result = vectors.get_sparse_vector(10)

    assert sparse.isspmatrix_coo(result), (
        "get_sparse_vector має повертати sparse.coo_matrix"
    )
    assert result.shape == (10, 1), (
        f"Очікувалася форма (10, 1), отримано {result.shape}"
    )


def test_addition_and_input_immutability():
    # Перевіряємо додавання та те, що функція не змінює вихідні вектори
    x = np.array([[1.0], [2.0], [-3.0]])
    y = np.array([[4.0], [-5.0], [6.0]])
    x_before = x.copy()
    y_before = y.copy()

    result = vectors.add(x, y)

    np.testing.assert_allclose(result, [[5.0], [-3.0], [3.0]])
    np.testing.assert_allclose(x, x_before)
    np.testing.assert_allclose(y, y_before)


def test_scalar_multiplication():
    # Перевіряємо множення на від'ємний та дробовий скаляр
    x = np.array([[2.0], [-4.0], [6.0]])

    result = vectors.scalar_multiplication(x, -0.5)

    np.testing.assert_allclose(result, [[-1.0], [2.0], [-3.0]])


def test_linear_combination_with_zero_and_negative_coefficients():
    # Перевіряємо кілька векторів, нульовий та від'ємний коефіцієнти
    x = np.array([[1.0], [2.0]])
    y = np.array([[3.0], [-1.0]])
    z = np.array([[-2.0], [5.0]])

    result = vectors.linear_combination([x, y, z], [2.0, -0.5, 0.0])

    # 2*x - 0.5*y + 0*z = [[0.5], [4.5]]
    np.testing.assert_allclose(result, [[0.5], [4.5]])


def test_dot_product_is_scalar_and_correct():
    # Перевіряємо значення скалярного добутку та те, що результат не є масивом (1, 1)
    x = np.array([[3.0], [4.0]])
    y = np.array([[4.0], [-3.0]])

    result = vectors.dot_product(x, y)

    assert np.isscalar(result), "dot_product має повертати число, а не масив"
    assert np.isclose(result, 0.0), f"Очікувалося 0, отримано {result}"


def test_norm_respects_order_for_negative_values():
    # Перевіряємо, що L1, L2 та нескінченна норма й справді різні
    # і рахуються за модулем координат
    x = np.array([[-3.0], [4.0]])

    assert np.isclose(vectors.norm(x, 1), 7.0), "Неправильна L1-норма"
    assert np.isclose(vectors.norm(x, 2), 5.0), "Неправильна L2-норма"
    assert np.isclose(vectors.norm(x, np.inf), 4.0), "Неправильна max-норма"


def test_norm_also_works_for_one_dimensional_vector():
    # Змодел. прихований випадок: вектор може бути (n,), а не лише стовпцем (n, 1).
    x = np.array([-3.0, 4.0])

    assert np.isclose(vectors.norm(x, 1), 7.0)
    assert np.isclose(vectors.norm(x, 2), 5.0)
    assert np.isclose(vectors.norm(x, np.inf), 4.0)


def test_distance_zero_and_symmetry():
    # Перевіряємо відстань вектора до себе та симетричність відстані
    x = np.array([[3.0], [4.0]])
    y = np.array([[0.0], [0.0]])

    assert np.isclose(vectors.distance(x, x), 0.0), (
        "Відстань вектора до себе має дорівнювати нулю"
    )
    assert np.isclose(vectors.distance(x, y), 5.0), "Неправильна L2-відстань"
    assert np.isclose(vectors.distance(x, y), vectors.distance(y, x)), (
        "Відстань має бути симетричною"
    )


def test_angles_for_parallel_orthogonal_and_opposite_vectors():
    # Перевіряємо три кути: 0, 90 та 180 градусів
    x = np.array([[1.0], [0.0]])
    same_direction = np.array([[8.0], [0.0]])
    orthogonal = np.array([[0.0], [-3.0]])
    opposite = np.array([[-2.0], [0.0]])

    assert np.isclose(vectors.cos_between_vectors(x, same_direction), 0.0)
    assert np.isclose(vectors.cos_between_vectors(x, orthogonal), 90.0)
    assert np.isclose(vectors.cos_between_vectors(x, opposite), 180.0)


def test_is_orthogonal():
    # Перевіряємо як перпендикулярні, так і неперпендикулярні вектори
    x = np.array([[3.0], [4.0]])
    perpendicular = np.array([[4.0], [-3.0]])
    not_perpendicular = np.array([[1.0], [1.0]])

    assert vectors.is_orthogonal(x, perpendicular) is True
    assert vectors.is_orthogonal(x, not_perpendicular) is False


def test_linear_system_solution_satisfies_equation():
    # Перевіряємо головну властивість розв'язку: A @ x має дорівнювати b
    a = np.array([[2.0, 1.0], [1.0, -1.0]])
    b = np.array([5.0, 1.0])

    result = vectors.solves_linear_systems(a, b)

    np.testing.assert_allclose(a @ result, b)


def test_linear_system_with_column_b():
    # Прихований випадок: b може бути не одномірним, а вектором-стовпцем
    a = np.array([[3.0, 1.0], [1.0, 2.0]])
    b = np.array([[9.0], [8.0]])

    result = vectors.solves_linear_systems(a, b)

    np.testing.assert_allclose(a @ result, b)


tests = [
    test_get_vector_shape_and_type,
    test_get_sparse_vector_shape_and_format,
    test_addition_and_input_immutability,
    test_scalar_multiplication,
    test_linear_combination_with_zero_and_negative_coefficients,
    test_dot_product_is_scalar_and_correct,
    test_norm_respects_order_for_negative_values,
    test_norm_also_works_for_one_dimensional_vector,
    test_distance_zero_and_symmetry,
    test_angles_for_parallel_orthogonal_and_opposite_vectors,
    test_is_orthogonal,
    test_linear_system_solution_satisfies_equation,
    test_linear_system_with_column_b,
]

failed = []

for test in tests:
    try:
        test()
        print(f"PASS: {test.__name__}")
    except Exception as error:
        failed.append((test.__name__, str(error)))
        print(f"FAIL: {test.__name__}\n  {error}")

if failed:
    raise AssertionError(f"\nНе пройдено тестів: {len(failed)}")

print(f"\nУсі {len(tests)} локальних тестів пройдено.")

PASS: test_get_vector_shape_and_type
PASS: test_get_sparse_vector_shape_and_format
PASS: test_addition_and_input_immutability
PASS: test_scalar_multiplication
PASS: test_linear_combination_with_zero_and_negative_coefficients
PASS: test_dot_product_is_scalar_and_correct
PASS: test_norm_respects_order_for_negative_values
PASS: test_norm_also_works_for_one_dimensional_vector
PASS: test_distance_zero_and_symmetry
PASS: test_angles_for_parallel_orthogonal_and_opposite_vectors
PASS: test_is_orthogonal
PASS: test_linear_system_solution_satisfies_equation
PASS: test_linear_system_with_column_b

Усі 13 локальних тестів пройдено.


# Код для vectors.pt

In [ ]:
from typing import Sequence

import numpy as np
from scipy import sparse


def get_vector(dim: int) -> np.ndarray:
    """Create random column vector with dimension dim.

    Args:
        dim (int): vector dimension.

    Returns:
        np.ndarray: column vector.
    """
    # raise NotImplementedError
    return np.random.random((dim, 1))

def get_sparse_vector(dim: int) -> sparse.coo_matrix:
    """Create random sparse column vector with dimension dim.

    Args:
        dim (int): vector dimension.

    Returns:
        sparse.coo_matrix: sparse column vector.
    """
    #raise NotImplementedError
    # Створюємо розріджений вектор-стовпець формату COO із випадковими числами (заповненість 25% -- випад. числа, інш. нулі)
    return sparse.random(dim, 1, density=0.25, format="coo")

def add(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Vector addition.

    Args:
        x (np.ndarray): 1st vector.
        y (np.ndarray): 2nd vector.

    Returns:
        np.ndarray: vector sum.
    """
    #raise NotImplementedError
    return x + y

def scalar_multiplication(x: np.ndarray, a: float) -> np.ndarray:
    """Vector multiplication by scalar.

    Args:
        x (np.ndarray): vector.
        a (float): scalar.

    Returns:
        np.ndarray: multiplied vector.
    """
    #raise NotImplementedError
    return x * a

def linear_combination(vectors: Sequence[np.ndarray], coeffs: Sequence[float]) -> np.ndarray:
    """Linear combination of vectors.

    Args:
        vectors (Sequence[np.ndarray]): list of vectors of len N.
        coeffs (Sequence[float]): list of coefficients of len N.

    Returns:
        np.ndarray: linear combination of vectors.
    """
    #raise NotImplementedError
    vectors_matrix = np.column_stack(vectors)
    # нижче перетворюємо масив коефіцієнтів на вертикальний вектор-стовпець (додаємо нову вісь)
    coeffs_array = np.array(coeffs)[:, np.newaxis]
    return vectors_matrix @ coeffs_array

def dot_product(x: np.ndarray, y: np.ndarray) -> float:
    """Vectors dot product.

    Args:
        x (np.ndarray): 1st vector.
        y (np.ndarray): 2nd vector.

    Returns:
        float: dot product.
    """
    # raise NotImplementedError
    # Перетвор. матриці на вектори на площині (.ravel), захищає якщо отрим. комплексн. чисела та строго повертає тип float
    return float(np.dot(x.ravel(), y.ravel()))

def norm(x: np.ndarray, order: int | float) -> float:
    """Vector norm: Manhattan, Euclidean or Max.

    Args:
        x (np.ndarray): vector
        order (int | float): norm's order: 1, 2 or inf.

    Returns:
        float: vector norm
    """
    #raise NotImplementedError
    # "Випрямляємо" масив у вектор на площині (.ravel) + обчислюємо задану векторну норму та строго повертає тип float
    return float(np.linalg.norm(x.ravel(), ord=order))

def distance(x: np.ndarray, y: np.ndarray) -> float:
    """L2 distance between vectors.

    Args:
        x (np.ndarray): 1st vector.
        y (np.ndarray): 2nd vector.

    Returns:
        float: distance.
    """
    #raise NotImplementedError
    # Перетворюємо різницю у вектор на площині як й вище + знаходимо Евклідову відстань L2 та строго повертає float
    return float(np.linalg.norm((x - y).ravel()))

def cos_between_vectors(x: np.ndarray, y: np.ndarray) -> float:
    """Angle between two vectors, in degrees.

    Despite the name, this returns the angle itself and not its cosine:
    0 for parallel vectors, 90 for orthogonal ones, 180 for opposite ones.

    Args:
        x (np.ndarray): 1st vector, shape (n, 1).
        y (np.ndarray): 2nd vector, shape (n, 1).

    Returns:
        float: angle in degrees, in [0, 180].
    """
    #raise NotImplementedError
    cosbw = dot_product(x, y) /(norm(x, 2)* norm(y, 2))
    # Обмеження через np.clip захищає нас від NaN через похибку float
    cosbw = np.clip(cosbw, -1.0, 1.0)
    # arccos знаходить кут, який конвертується в градуси
    return float(np.degrees(np.arccos(cosbw)))


def is_orthogonal(x: np.ndarray, y: np.ndarray) -> bool:
    """Check is vectors orthogonal.

    Args:
        x (np.ndarray): 1st vector.
        y (np.ndarray): 2nd vector.


    Returns:
        bool: are vectors orthogonal.
    """
    #raise NotImplementedError
    # Перевіряємо ортогон-сть з урахуванням похибки float (np.isclose)
    # + строго повертає стандартний тип bool
    return bool(np.isclose(dot_product(x, y), 0.0))

def solves_linear_systems(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Solve system of linear equations.

    Args:
        a (np.ndarray): coefficient matrix.
        b (np.ndarray): ordinate values.

    Returns:
        np.ndarray: sytems solution
    """
    #raise NotImplementedError
    return np.linalg.solve(a, b)